# Week 57, Day 4: The dual, support vectors and kernel SVMs

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w057/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Dual coordinate ascent

Complete svm_dual: for each i in turn, compute the gradient of the dual, g_i = y_i f(x_i) - 1 (with f maintained incrementally), take the Newton step alpha_i - g_i / K_ii, clip it to [0, C], and update f. Run it with a linear kernel plus 1 on overlapping data, then check the KKT conditions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(574)

def svm_dual(K, Y, C, sweeps=100):
    n = len(Y); alpha = np.zeros(n); f = np.zeros(n)          # f = K (alpha * Y)
    for s in range(sweeps):
        for i in range(n):
            g = Y[i] * f[i] - 1
            new = ...
            d = new - alpha[i]
            if d != 0:
                f += d * Y[i] * K[:, i]
                alpha[i] = new
    return alpha, f

Y = np.repeat([-1.0, 1.0], 100); X = r.normal(size=(200, 2)) + np.outer(Y, [0.9, 0.5])
Yt = np.repeat([-1.0, 1.0], 1000); Xt = r.normal(size=(2000, 2)) + np.outer(Yt, [0.9, 0.5])
C = 1.0
K = X @ X.T + 1
alpha, f_train = svm_dual(K, Y, C)
w = (alpha * Y) @ X; b = np.sum(alpha * Y)                       # the bias folded into the kernel
acc_lin = np.mean(np.sign(Xt @ w + b) == Yt)
print(f"support vectors: {np.sum(alpha > 1e-8)} of 200 ({np.sum(alpha > C - 1e-8)} at the bound C)")
print(f"w = {np.round(w, 3)}, b = {b:.3f}; test accuracy {acc_lin:.3f}")

In [ ]:
import numpy as np
def test_box():
    "every alpha lies in [0, C]"
    assert np.all(alpha >= -1e-12) and np.all(alpha <= C + 1e-12)
def test_kkt():
    "zero alphas have margin at least 1 (within 0.05) and alphas at C have margin at most 1"
    m = Y * f_train
    assert np.all(m[alpha < 1e-8] > 1 - 0.05) and np.all(m[alpha > C - 1e-8] < 1 + 0.05)
def test_acc():
    "the linear SVM's test accuracy is above 0.8"
    assert acc_lin > 0.8

test_box()
test_kkt()
test_acc()
print("All checks passed")

### Exercise: Two crescents

Make two interleaved crescents (200 training and 2,000 test examples). Train the dual SVM with a linear kernel and with a Gaussian kernel (gamma = 2, C = 10), each with 1 added for the bias, and compare test accuracy and the number of support vectors.

In [ ]:
def moons(n, g):
    y = np.where(g.random(n) < 0.5, -1.0, 1.0); a = np.pi * g.random(n)
    x = np.c_[np.where(y > 0, np.cos(a), 1 - np.cos(a)), np.where(y > 0, np.sin(a), 0.5 - np.sin(a))]
    return x + 0.15 * g.normal(size=(n, 2)), y

def rbf(A, B, gamma):
    d2 = (A ** 2).sum(1)[:, None] - 2 * A @ B.T + (B ** 2).sum(1)[None, :]
    return np.exp(-gamma * np.maximum(d2, 0))

Xm, Ym = moons(200, r); Xmt, Ymt = moons(2000, r)
a_lin, _ = svm_dual(Xm @ Xm.T + 1, Ym, 10.0)
acc_lin_moons = np.mean(np.sign(((Xmt @ Xm.T + 1) @ (a_lin * Ym))) == Ymt)
a_rbf, _ = svm_dual(rbf(Xm, Xm, 2.0) + 1, Ym, 10.0)
acc_rbf = ...
n_sv = int(np.sum(a_rbf > 1e-8))
print(f"linear kernel: test accuracy {acc_lin_moons:.3f}; Gaussian kernel: {acc_rbf:.3f} with {n_sv} support vectors of 200")

In [ ]:
import numpy as np
def test_moons():
    "the Gaussian-kernel SVM is far more accurate than the linear one on the crescents"
    assert acc_rbf > 0.95 and acc_rbf > acc_lin_moons + 0.08
def test_sv():
    "support vectors are a minority of the training set"
    assert n_sv < 100

test_moons()
test_sv()
print("All checks passed")

### Exercise: A cross-validated grid

For each C and gamma on the grids, compute three-fold cross-validated accuracy on the 200 training crescents. Refit the best pair on all training data and report test accuracy.

In [ ]:
Cs, gammas = [0.1, 1, 10, 100], [0.05, 0.5, 2, 10, 100]
folds = np.array_split(r.permutation(200), 3)
cv_acc = np.zeros((len(Cs), len(gammas)))
for i, C_ in enumerate(Cs):
    for j, gm in enumerate(gammas):
        correct = 0
        for te in folds:
            tr = np.setdiff1d(np.arange(200), te)
            a, _ = svm_dual(rbf(Xm[tr], Xm[tr], gm) + 1, Ym[tr], C_, sweeps=40)
            correct += ...
        cv_acc[i, j] = correct / 200
i, j = np.unravel_index(np.argmax(cv_acc), cv_acc.shape)
a, _ = svm_dual(rbf(Xm, Xm, gammas[j]) + 1, Ym, Cs[i])
acc_best = np.mean(np.sign((rbf(Xmt, Xm, gammas[j]) + 1) @ (a * Ym)) == Ymt)
print(np.round(cv_acc, 3))
print(f"best: C = {Cs[i]}, gamma = {gammas[j]}; test accuracy {acc_best:.3f}")

In [ ]:
import numpy as np
def test_grid():
    "cv_acc has one entry per C and gamma"
    assert cv_acc.shape == (len(Cs), len(gammas))
def acc_best():
    "the best cross-validated pair reaches test accuracy above 0.95, and the extreme gamma values do worse in cross-validation"
    assert acc_best > 0.95 and cv_acc[:, 0].max() < cv_acc.max() and cv_acc[:, -1].max() < cv_acc.max()

test_grid()
print("All checks passed")